# Contagion Simulations

For every bank in every quarter (2016 Q1 – 2023 Q4), apply fixed equity shocks
via `simulate_failure` and store the cascade results as three normalised tables.

| Output directory | Table | Grain |
|---|---|---|
| `src/data/sim_runs/` | **Table 1 – runs** | 1 row per simulation |
| `src/data/sim_bank_state/` | **Table 2 – bank_end_state_sparse** | 1 row per impacted / initial / failed bank |
| `src/data/sim_round_summary/` | **Table 3 – round_summary** | 1 row per cascade round (optional) |

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_data, build_run_row, build_bank_rows, build_round_rows
from src.models import simulate_failure

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
N_SIMULATIONS   = 500_000   # simulations per quarter
MECHANISM       = "Exposure"
ALPHA           = 1.0
SPREAD_WO_DEF   = True
SHOCK_MIN       = 0.05
SHOCK_MAX       = 0.60

# Set to True to also write Table 3 (round_summary).
# Enabling this adds ~15-20 % to runtime per quarter.
TRACK_ROUNDS    = False

PROJECT_ROOT    = Path().resolve().parent

OUT_RUNS        = PROJECT_ROOT / "src" / "data" / "sim_runs"
OUT_BANK_STATE  = PROJECT_ROOT / "src" / "data" / "sim_bank_state"
OUT_ROUND_SUM   = PROJECT_ROOT / "src" / "data" / "sim_round_summary"

for d in (OUT_RUNS, OUT_BANK_STATE, OUT_ROUND_SUM):
    d.mkdir(parents=True, exist_ok=True)

print("Output dirs:")
print(" ", OUT_RUNS)
print(" ", OUT_BANK_STATE)
print(" ", OUT_ROUND_SUM)

In [ ]:
# Load all 32 quarters
quarters = {}

for year in range(2016, 2024):
    for q in range(1, 5):
        edges, nodes = load_data(year, q)
        quarters[(year, q)] = (edges, nodes)

In [ ]:
# ---------------------------------------------------------------------------
# Main simulation loop
# ---------------------------------------------------------------------------
summary_rows = []

for (year, q), (edges, nodes) in quarters.items():

    # ---- per-quarter constants (precomputed once) -------------------------
    equity_initial = nodes.set_index("index")["Equity"].to_dict()
    n_banks        = len(nodes)
    n_edges        = len(edges)
    bank_ids       = nodes["index"].tolist()

    # Seeded RNG per quarter for reproducibility
    rng            = np.random.default_rng(seed=year * 10 + q)
    sampled_banks  = rng.choice(bank_ids, size=N_SIMULATIONS, replace=True)
    shock_fracs    = rng.uniform(SHOCK_MIN, SHOCK_MAX, size=N_SIMULATIONS)

    # Batch timestamp (one value per quarter, not per simulation)
    batch_ts = datetime.now(timezone.utc).isoformat()

    run_rows   = []
    bank_rows  = []
    round_rows = []

    for bank_id, shock_frac in zip(sampled_banks, shock_fracs):
        run_id = str(uuid.uuid4())

        t0 = time.perf_counter()
        result = simulate_failure(
            int(bank_id), edges, nodes,
            mechanism=MECHANISM,
            alpha=ALPHA,
            spread_without_default=SPREAD_WO_DEF,
            initial_loss_mode="fixed",
            initial_loss_frac=float(shock_frac),
            track_rounds=TRACK_ROUNDS,
        )
        runtime_ms = (time.perf_counter() - t0) * 1_000

        run_rows.append(build_run_row(
            run_id, result, equity_initial, n_banks, n_edges,
            initial_bank=int(bank_id),
            mechanism=MECHANISM,
            alpha=ALPHA,
            spread_without_default=SPREAD_WO_DEF,
            initial_loss_mode="fixed",
            year=year, quarter=q,
            runtime_ms=runtime_ms,
            timestamp_utc=batch_ts,
        ))

        bank_rows.extend(build_bank_rows(
            run_id, result, equity_initial, int(bank_id)
        ))

        if TRACK_ROUNDS:
            round_rows.extend(build_round_rows(run_id, result))

    # ---- save Table 1 (runs) ---------------------------------------------
    df_runs = pd.DataFrame(run_rows)
    out_runs = OUT_RUNS / f"sim_runs_{year}Q{q}.parquet"
    df_runs.to_parquet(out_runs, index=False)

    # ---- save Table 2 (bank_end_state_sparse) ----------------------------
    df_banks = pd.DataFrame(bank_rows)
    out_banks = OUT_BANK_STATE / f"sim_bank_state_{year}Q{q}.parquet"
    df_banks.to_parquet(out_banks, index=False)

    # ---- save Table 3 (round_summary, optional) --------------------------
    if TRACK_ROUNDS and round_rows:
        df_rounds = pd.DataFrame(round_rows)
        out_rounds = OUT_ROUND_SUM / f"sim_round_summary_{year}Q{q}.parquet"
        df_rounds.to_parquet(out_rounds, index=False)
        round_info = f" | {len(df_rounds):,} round rows"
    else:
        round_info = ""

    summary_rows.append({
        "year":                   year,
        "quarter":                q,
        "period":                 f"{year}Q{q}",
        "n_simulations":          N_SIMULATIONS,
        "n_bank_rows":            len(df_banks),
        "pct_initial_default":    df_runs["initial_default"].mean(),
        "avg_cascade_failed":     df_runs["num_failed"].mean(),
        "avg_impacted_share":     df_runs["impacted_share"].mean(),
        "avg_sys_eq_depletion":   df_runs["system_equity_depletion"].mean(),
    })

    print(f"[OK] {year}Q{q} | {N_SIMULATIONS:,} runs | {len(df_banks):,} bank rows{round_info}")

df_summary = (
    pd.DataFrame(summary_rows)
      .sort_values(["year", "quarter"])
      .reset_index(drop=True)
)
display(df_summary)

## Verification

In [ ]:
# ---- Table 1 spot-check --------------------------------------------------
df_runs_check = pd.read_parquet(OUT_RUNS / "sim_runs_2016Q1.parquet")
print("Table 1 – runs")
print(f"  shape : {df_runs_check.shape}")
print(f"  cols  : {list(df_runs_check.columns)}")
display(df_runs_check.head(3))
display(df_runs_check[[
    "shock_fraction", "initial_default", "rounds",
    "num_failed", "num_impacted", "failed_share",
    "system_equity_depletion", "max_bank_loss",
]].describe())

In [ ]:
# ---- Table 2 spot-check --------------------------------------------------
df_banks_check = pd.read_parquet(OUT_BANK_STATE / "sim_bank_state_2016Q1.parquet")
print("Table 2 – bank_end_state_sparse")
print(f"  shape : {df_banks_check.shape}")
print(f"  cols  : {list(df_banks_check.columns)}")
display(df_banks_check.head(5))
display(df_banks_check[["equity_initial", "equity_final", "equity_loss",
                         "fail_round", "loss_frac_of_initial"]].describe())

In [ ]:
# ---- Cross-table consistency check ---------------------------------------
# For every run: Table 1 num_failed  ==  Table 2 rows where failed=True
t1 = df_runs_check[["run_id", "num_failed"]].copy()
t2_failed = (
    df_banks_check[df_banks_check["failed"]]
    .groupby("run_id")
    .size()
    .rename("t2_failed_count")
    .reset_index()
)
check = t1.merge(t2_failed, on="run_id", how="left")
check["t2_failed_count"] = check["t2_failed_count"].fillna(0).astype(int)

mismatches = check[check["num_failed"] != check["t2_failed_count"]]
print(f"Cross-table mismatches: {len(mismatches)}  (expect 0)")
assert len(mismatches) == 0, "Tables 1 and 2 are inconsistent!"